In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.

def get_json(url):
    response=requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


Northern Star Shop API


In [0]:
#2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи

def fetch_all(first_url):
    records=[]
    url=first_url
    while url:
        page=get_json(url)
        records.extend(page['data'])
        url=urljoin(url, page['next']) if page.get('next') else None
    return records
    
# print(len(fetch_all(BASE_URL + 'orders/page-1.json'))) #1852       

1852


In [0]:
#3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->None која листу речника уписује у JSON фајл без икаквих измена.

def save_json(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    

None


In [0]:
#4
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes, unchanged
    return len(response.content)    
#print(download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))) #801

801


In [0]:
#5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams', 'matches'

def extract(raw_dir):
    orders=fetch_all(BASE_URL + 'orders/page-1.json') 
    save_json(orders, os.path.join(raw_dir, 'orders.json'))

    for n in ['products.csv', 'teams.csv', 'matches.parquet']:
        download_file(FILES_URL+n, os.path.join(raw_dir, n))

    orders_df = pd.json_normalize(orders, sep='_')  
    orders_df = orders_df.rename(columns={'customer_name':'customer', 'customer_city': 'city'}) 

    return {
        'orders': orders_df,
        'products': pd.read_csv(os.path.join(raw_dir, 'products.csv')),
        'teams': pd.read_csv(os.path.join(raw_dir, 'teams.csv')),
        'matches': pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))
    } 
    

{'orders':          order_id  order_date  ...           customer        city
0       NSS-00001  2026-08-15  ...  NIKOLA STOJANOVIĆ   Zrenjanin
1      nss-00002   2026-08-15  ...   Jovana Pavlović        Čačak
2       NSS-00003  2026-08-15  ...      Sara Živković     Beograd
3       NSS-00004  2026-08-15  ...      Sara Đorđević   Subotica 
4       NSS-00005  2026-08-15  ...     Stefan Popović         Niš
...           ...         ...  ...                ...         ...
1847    NSS-01836  2026-11-15  ...     MILICA POPOVIĆ   SUBOTICA 
1848    NSS-01837  2026-11-15  ...    marko kovačević       Čačak
1849    NSS-01838  2026-11-15  ...      Marko Nikolić      Čačak 
1850    NSS-01839  2026-11-15  ...    Jelena Živković   Zrenjanin
1851    NSS-01840  2026-11-15  ...     ĐORĐE ĐORĐEVIĆ    Novi Sad

[1852 rows x 8 columns], 'products':      product_id           sku     product_name  ...       category     price active
0          1001  NPM-JER-1001  NPM Home Jersey  ...        jerseys   7490.0

In [0]:
#6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа

def clean_products(df):
    df=df.drop_duplicates(subset='product_id', keep='first').copy
    df['team_code']=df['team_code'].str.strip().upper()
    df['category']=df['category'].str.strip().lower()
    price = df['price'].astype(str).str.strip().str.replace(' RSD', '', regex=False).str.replace(',', '.', regex=False)
    df['price']= pd.to_numeric(price, errors='coerce').astype(float)
    df['active']=df['active'].astype(str).str.strip().str.lower()=='yes'
    return df


In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame

def clean_orders(df):
    df=df.copy()
    for col in ['order_id', 'customer', 'city', 'payment']:
        df[col]=df[col].str.strip()
    df['order_id']=df['order_id'].str.upper()
    df['customer']=df['customer'].str.title()
    df['city']=df['city'].str.title().fillna('Unknown')
    return df    

In [0]:
# #8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df):
    is_dup=df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа 'OK' за исправан ред, у супротном разлог

def validate_order(row, product_ids):
    quantity=pd.to_numeric(row['quantity'], errors='coerce')
    price=pd.to_numeric(row['unit_price'], errors='coerce')
    product_id=pd.to_numeric(row['product_id'],errors='coerce')
    if not str(row['order_id']).startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <=0 or quantity!=int(quantity):
        return 'invalid_quantity'
    if pd.isna(price) or price <=0:
        return 'invalid_price'
    if pd.isna(product_id) or int(product_id) not in product_ids:
        return 'unknown_product'
    return 'OK'



In [0]:
#10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->pd.DataFrame

def transform_orders(df, products):
    df=df.copy()
    df['quantity']=pd.to_numeric(df['quantity']).astype(int)
    df['unit_price']=pd.to_numeric(df['unit_price']).astype(float)
    df['order_date']= pd.to_datetime(df['order_date'])
    df['total'] = df['quantity']*df['unit_price']
    names=products[['product_id', 'product_name', 'team_code', 'category']]
    return df.merge(names, on='product_id', how='left')

In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected)
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df, products):
    df=clean_orders(df)
    unique, duplicates=remove_duplicate_orders(df)
    duplicates=duplicates.assign(reason='duplicate')
    product_ids=set(products['product_id'])
    reasons=unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid=transform_orders(unique[reasons=='OK'], products)
    rejected=pd.concat([unique['reasons!=OK'].assign(reason=reasons['reasons'!='OK']), duplicates])
    return valid, rejected



---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-5992993530580482>, line 18
     15     return valid, rejected
     17 data = extract(RAW_DIR)
---> 18 products = clean_products(data['products'])
     19 valid, rejected = split_orders(data['orders'], products)
     20 print(len(valid), len(rejected))

File <command-5992993530580475>, line 15, in clean_products(df)
     13 def clean_products(df):
     14     df=df.drop_duplicates(subset='product_id', keep='first').copy
---> 15     df['team_code']=df['team_code'].str.strip().upper()
     16     df['category']=df['category'].str.strip().lower()
     17     price = df['price'].astype(str).str.strip().str.replace(' RSD', '', regex=False).str.replace(',', '.', regex=False)

TypeError: 'method' object is not subscriptable

In [0]:
#17. Load. Написати функцију load(outputs: dict, processed_dir: str) -> dict која сваки DataFrame из речника {име_фајла: DataFrame} уписује у processed_dir и враћа {име_фајла: број_редова}
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:    
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written    


In [0]:
#18. Написати функцију run_pipeline(raw_dir: str, processed_dir: str) -> dict која покреће цео pipeline
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
        }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
        }
    pd.DataFrame(list(summary.items()), columns=['metric','value']).to_csv(os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary


In [0]:
run_pipeline(RAW_DIR, PROCESSED_DIR)